# Proyecto Loti Perú — Pipeline MLOps
> Notebooks listos para Databricks. Ajustá la variable `DATA_PATH` para tu ruta.

## 04 · Auditoría y Gobernanza (Trazabilidad + Calidad)

In [0]:
CATALOG = "main"
SCHEMA  = "loterias_silver"
TABLE_SILVER   = f"{CATALOG}.{SCHEMA}.apuestas_silver"
TABLE_FEATURES = f"{CATALOG}.{SCHEMA}.features_apuestas"

# 1) Trazabilidad Delta (time travel / history)
hist_silver = spark.sql(f"DESCRIBE HISTORY {TABLE_SILVER}")
display(hist_silver)

hist_feat = spark.sql(f"DESCRIBE HISTORY {TABLE_FEATURES}")
display(hist_feat)

### 2) Reglas rápidas de calidad (Great Expectations placeholder o PySpark checks)

In [0]:
from pyspark.sql import functions as F

df = spark.table(TABLE_SILVER)

checks = {
    "monto_pos": df.filter(F.col("monto") <= 0).count() == 0,
    "monto_max": df.filter(F.col("monto") >= 10000).count() == 0,
    "tx_id_unicos": df.select("tx_id").distinct().count() == df.count(),
    "sin_nulos_user": df.filter(F.col("user_id").isNull()).count() == 0,
}

print("Resultados de controles:", checks)

# Si algo falla, puedes lanzar excepción para abortar pipeline/Job
if not all(checks.values()):
    raise ValueError("Falla en controles de calidad. Revisar 'checks'.")

### 3) Evidencias y responsables (bitácora simple)

In [0]:
from datetime import datetime
import json

log = {
    "datetime": datetime.utcnow().isoformat(),
    "dataset": TABLE_SILVER,
    "owner": "auditoria@datateam.local",
    "checks": checks,
    "notes": "Controles básicos de rango, nulos y duplicados. Ver history para trazabilidad."
}
display(spark.createDataFrame([(json.dumps(log),)], ["audit_log_json"]))